In [ ]:
import unsloth
from unsloth import FastLanguageModel
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq
from datasets import load_dataset
import os
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_properties(0).total_memory / 1024**3)

# Ensure we save in the right directory
os.makedirs("ollama_phi_chat", exist_ok=True)

# 2. Load Phi-4-mini-instruct model (publicly available on HuggingFace)
print("Loading Phi-4-3-mini-4k-instruct model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",  
    torch_dtype="auto",  
    trust_remote_code=True, 
)



In [ ]:
# 3. Apply LoRA
print("Applying LoRA...")
model = FastLanguageModel.get_peft_model(
    model,
    r=8,                                 # reduced from 16
    lora_alpha=16,                       # reduced from 32
    lora_dropout=0.05,
    target_modules=["q_proj",
                    "k_proj",
                    "v_proj",
                    "o_proj",
                    "gate_proj",
                    "up_proj",
                    "down_proj"]
)

In [ ]:
# 4. Format data for instruction tuning
print("Loading and formatting dataset...")
dataset = load_dataset("json", data_files="dotnet_azure_ai_dataset.json")

# Use only first 200 samples for testing (even faster)
train_data = dataset["train"]
split_data = train_data.train_test_split(test_size=0.1)

def format_instruction(example):
    # Format as instruction following
    messages = example["messages"]
    text = ""
    for msg in messages:
        text += f"<|{msg['role']}|>\n{msg['content']}\n"
    text += tokenizer.eos_token
    return {"text": text}

In [ ]:
# Apply formatting
train_dataset = split_data["train"].map(format_instruction)
test_dataset = split_data["test"].map(format_instruction)

# Tokenize
def tokenize_function(examples):
    model_inputs = tokenizer(
        examples["text"],
        truncation=True,
        max_length=1024,
        padding="max_length",
    )
    # Labels should be the same as input_ids for causal LM training
    model_inputs["labels"] = model_inputs["input_ids"].copy()
    return model_inputs

train_dataset = train_dataset.map(tokenize_function, remove_columns=train_dataset.column_names)# ✅ remove everything except tensors
test_dataset = test_dataset.map(tokenize_function, remove_columns=test_dataset.column_names)# ✅ remove everything except tensors

# print(f"Training samples: {len(train_dataset)}")
# print(f"Test samples: {len(test_dataset)}")

In [ ]:
torch.cuda.empty_cache()
# 5. Train with smaller settings
print("Starting training (this may take a few minutes)...")
try:
    training_args = TrainingArguments(
        output_dir="phichat",
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,      # increased
        warmup_steps=5,
        logging_steps=1,
        eval_strategy="no",                  # DISABLE eval — biggest VRAM saver
        save_strategy="epoch",
        save_total_limit=1,
        fp16=False,
        bf16=True,
        remove_unused_columns=False,
        report_to="none",
        optim="adamw_8bit",                  # 8-bit optimizer saves ~1GB
        max_grad_norm=0.3,
        dataloader_pin_memory=False,         # saves CPU→GPU transfer memory
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8),
    )

    trainer.train()
    print("✓ Training completed successfully!")

except Exception as e:
    print(f"✗ Training error: {type(e).__name__}")
    print(f"Error message: {str(e)}")
    import traceback
    traceback.print_exc()

# 6. Save model
print("Saving model...")
model.save_pretrained("ollama_phi_chat")
tokenizer.save_pretrained("ollama_phi_chat")

print("✓ Model saved successfully to ollama_phi_chat/")
print("\nFiles saved:")
for file in os.listdir("ollama_phi_chat"):
    print(f"  - {file}")